cd Desktop/Coding\ project/StockAnalysis ; git add . ; git commit -m "In progress" ; git push origin main

git repack -a -d -f --depth=250 --window=250

git reflog expire --all --expire=now ; git gc --prune=now --aggressive

In [ ]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_excel_filename, get_infix, get_volume5m_data, generate_end_dates, merge_stocks, stock_market
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import multiprocessing
import numpy as np
import pandas as pd
from pandas import ExcelWriter as EW
from plot import *
import random
from scipy.optimize import minimize
from scipy.stats import false_discovery_control, kendalltau, linregress, pearsonr, spearmanr, ttest_ind, wilcoxon
from stock_screener import check_conds_tech, check_conds_fund, EM_rating, get_stock_info, stoploss_target
from technicals import *
from tqdm import tqdm

# Connect to TradingView
from tvDatafeed import TvDatafeed, Interval
tv = TvDatafeed()

# Start of the program
start = dt.datetime.now()

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Modify the current date
current_date = modify_current_date(start, index_name)

In [ ]:
sp500_df = get_df("^GSPC", current_date, max_period=True)
bear_df, bull_df, bear_starts, bear_ends, bull_starts, bull_ends = find_market_cycles(sp500_df)

In [ ]:
def calculate_rs_sp500_drops(index_df, stock_df, period=60):
    """
    Calculate the relative strength of a stock during index drop days.

    Parameters:
    index_df: DataFrame with index price data.
    stock_df: DataFrame with individual stock price data.
    period (int, optional): Number of recent days to consider. Default is 60 days.

    Returns:
    dict: Contains SP500 drop sum, stock change sum, and relative strength ratio
    """

    # Get recent data based on period
    index_recent = index_df.tail(period).copy()
    stock_recent = stock_df.tail(period).copy()
    
    # Calculate daily percent changes
    index_recent["Percent Change"] = index_recent["Close"].pct_change()
    stock_recent["Percent Change"] = stock_recent["Close"].pct_change()

    # Find dates where index drops
    index_drop_dates = index_recent[index_recent["Percent Change"] < 0].index

    # Filter stock data for those same dates
    stock_on_index_drops = stock_recent[stock_recent.index.isin(index_drop_dates)]

    # Sum the percent changes
    index_drop_sum = index_recent[index_recent["Percent Change"] < 0]["Percent Change"].sum()
    stock_change_sum = stock_on_index_drops["Percent Change"].sum()

    return {
        "index_drop_sum": index_drop_sum,
        "stock_change_sum": stock_change_sum
    }

In [ ]:
# Create a figure with 4 subplots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))

# Bear market durations
ax1.hist(bear_df["Duration (days)"], bins=range(0, int(bear_df["Duration (days)"].max()) + 200, 200), alpha=0.7, color="red", edgecolor="black")
ax1.set_title("Distribution of Bear Market Durations")
ax1.set_xlabel("Duration (days)")
ax1.set_ylabel("Frequency")
median_bear_duration = bear_df["Duration (days)"].median()
ax1.axvline(median_bear_duration, color="black", linestyle=":", label="Median")
ax1.text(median_bear_duration + 50, ax1.get_ylim()[1] * 0.9, f'Median: {median_bear_duration:.0f} days', 
         verticalalignment="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
ax1.legend()

# Bear market max drops
bear_drops = [float(drop.strip("%")) for drop in bear_df["Max Drop (%)"]]
ax2.hist(bear_drops, bins=range(int(min(bear_drops)) - 10, int(max(bear_drops)) + 10, 10), alpha=0.7, color="red", edgecolor="black")
ax2.set_title("Distribution of Bear Market Max Drops")
ax2.set_xlabel("Max Drop (%)")
ax2.set_ylabel("Frequency")
median_bear_drops = np.median(bear_drops)
ax2.axvline(median_bear_drops, color="black", linestyle=":", label="Median")
ax2.text(median_bear_drops - 10, ax2.get_ylim()[1] * 0.9, f'Median: {median_bear_drops:.0f}%', 
         verticalalignment="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
ax2.legend()

# Bull market durations
ax3.hist(bull_df["Duration (days)"], bins=range(0, int(bull_df["Duration (days)"].max()) + 200, 200), alpha=0.7, color="green", edgecolor="black")
ax3.set_title("Distribution of Bull Market Durations")
ax3.set_xlabel("Duration (days)")
ax3.set_ylabel("Frequency")
median_bull_duration = bull_df["Duration (days)"].median()
ax3.axvline(median_bull_duration, color="black", linestyle=":", label="Median")
ax3.text(median_bull_duration + 50, ax3.get_ylim()[1] * 0.9, f'Median: {median_bull_duration:.0f} days', 
         verticalalignment="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
ax3.legend()

# Bull market total returns
bull_returns = [float(ret.strip("%")) for ret in bull_df["Total Return (%)"]]
ax4.hist(bull_returns, bins=range(0, int(max(bull_returns)) + 10, 10), alpha=0.7, color="green", edgecolor="black")
ax4.set_title("Distribution of Bull Market Total Returns")
ax4.set_xlabel("Total Return (%)")
ax4.set_ylabel("Frequency")
median_bull_returns = np.median(bull_returns)
ax4.axvline(median_bull_returns, color="black", linestyle=":", label="Median")
ax4.text(median_bull_returns + 10, ax4.get_ylim()[1] * 0.9, f'Median: {median_bull_returns:.0f}%', 
         verticalalignment="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
ax4.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Define the stocks to optimize
stocks = ["SPMO", "IDMO", "3070.HK"]

# Get price data for all stocks
dfs = [get_df(stock, current_date, adj=True, max_period=True) for stock in stocks]

# Merge all stock data
merged_df = merge_stocks(stocks, dfs)

# Calculate daily returns
returns_df = merged_df.pct_change(fill_method=None).dropna()[[f"Close ({stock})" for stock in stocks]]

# Calculate and display correlation matrix
plot_corr_stocks(stocks, dfs)

# Optimization functions
def portfolio_sharpe(weights, returns):
    portfolio_return = np.sum(returns.mean() * weights) * 252
    portfolio_variance = np.dot(weights.T, np.dot(returns.cov() * 252, weights))
    portfolio_std = np.sqrt(portfolio_variance)
    return -portfolio_return / portfolio_std

def portfolio_risk_parity(weights, returns):
    cov_matrix = returns.cov() * 252
    portfolio_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
    marginal_risk = np.dot(cov_matrix, weights) / np.sqrt(portfolio_variance)
    risk_contributions = weights * marginal_risk
    target_risk = portfolio_variance / len(weights)
    return np.sum((risk_contributions - target_risk) ** 2)

# Optimization setup
constraints = {"type": "eq", "fun": lambda x: np.sum(x) - 1}
bounds = tuple((0, 1) for _ in range(len(stocks)))
x0 = np.array([1 / len(stocks)] * len(stocks))

# Run optimizations
result_sharpe = minimize(portfolio_sharpe, x0, method="SLSQP", 
                        bounds=bounds, constraints=constraints, args=(returns_df,))
result_risk_parity = minimize(portfolio_risk_parity, x0, method="SLSQP", 
                             bounds=bounds, constraints=constraints, args=(returns_df,))

# Display results
strategies = {"Max Sharpe": result_sharpe.x, "Risk Parity": result_risk_parity.x}

for strategy_name, weights in strategies.items():
    portfolio_returns_strategy = (returns_df * weights).sum(axis=1)
    annual_return = portfolio_returns_strategy.mean() * 252
    annual_volatility = portfolio_returns_strategy.std() * np.sqrt(252)
    sharpe_ratio = annual_return / annual_volatility
    mdd = (portfolio_returns_strategy.cumsum() - portfolio_returns_strategy.cumsum().cummax()).min()
    print(f"\n{strategy_name} Portfolio:")
    for i, stock in enumerate(stocks):
        print(f"{stock}: {weights[i]:.2%}")
    print(f"Annual Return: {annual_return:.2%}")
    print(f"Annual Volatility: {annual_volatility:.2%}")
    print(f"Sharpe Ratio: {sharpe_ratio:.4f}")
    print(f"Max Drawdown: {mdd:.2%}")

    weighted_vol = np.sum(weights * returns_df.std() * np.sqrt(252))
    diversification_ratio = weighted_vol / annual_volatility
    print(f"Diversification Ratio: {diversification_ratio:.4f}")

# Individual stock performance
print("\nIndividual Stock Performance")
for i, stock in enumerate(stocks):
    stock_returns = returns_df.iloc[:, i]
    stock_cagr = (1 + stock_returns).prod() ** (252 / len(stock_returns)) - 1
    stock_volatility = stock_returns.std() * np.sqrt(252)
    stock_sharpe = stock_returns.mean() / stock_returns.std() * np.sqrt(252)
    stock_mdd = (stock_returns.cumsum() - stock_returns.cumsum().cummax()).min()
    print(f"{stock}:")
    print(f"CAGR: {stock_cagr:.2%}")
    print(f"Volatility: {stock_volatility:.2%}")
    print(f"Sharpe Ratio: {stock_sharpe:.4f}")
    print(f"Max Drawdown: {stock_mdd:.2%}")

# Store optimal weights for use in other cells
optimal_weights = result_sharpe.x
portfolio_returns = (returns_df * optimal_weights).sum(axis=1)

In [ ]:
# Get S&P 500 data for comparison
sp500_df = get_df("^GSPC", current_date, adj=True, max_period=True)

# Filter S&P 500 data to match the portfolio returns timeframe
sp500_filtered = sp500_df[sp500_df.index.isin(returns_df.index)]
sp500_returns = sp500_filtered["Close"].pct_change().dropna()

# Calculate cumulative returns for all strategies
max_sharpe_portfolio = (returns_df * strategies["Max Sharpe"]).sum(axis=1)
risk_parity_portfolio = (returns_df * strategies["Risk Parity"]).sum(axis=1)

# Calculate cumulative performance
max_sharpe_cumulative = (1 + max_sharpe_portfolio).cumprod()
risk_parity_cumulative = (1 + risk_parity_portfolio).cumprod()
sp500_cumulative = (1 + sp500_returns).cumprod()

# Create the performance comparison plot
plt.figure(figsize=(12, 8))
plt.plot(max_sharpe_cumulative.index, max_sharpe_cumulative, label="Max Sharpe Portfolio", linewidth=2)
plt.plot(risk_parity_cumulative.index, risk_parity_cumulative, label="Risk Parity Portfolio", linewidth=2)
plt.plot(sp500_cumulative.index, sp500_cumulative, label="S&P 500", linewidth=2)

plt.title("Portfolio Performance Comparison vs S&P 500", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Cumulative Return", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print final performance metrics
print("\nFinal Performance Comparison:")
print(f"Max Sharpe Portfolio: {max_sharpe_cumulative.iloc[-1]:.2f}")
print(f"Risk Parity Portfolio: {risk_parity_cumulative.iloc[-1]:.2f}")
print(f"S&P 500: {sp500_cumulative.iloc[-1]:.2f}")

In [ ]:
# Get the MMTH data
mmth_data = get_df("MMTH", current_date, method="tradingview")

# Get the MMFI data
mmfi_data = get_df("MMFI", current_date, method="tradingview")

In [ ]:
# Get the S&P 500 data
sp500_df = get_df("^GSPC", current_date, max_period=True)

# Find bear markets
bear_df, bull_df, bear_starts, bear_ends, bull_starts, bull_ends = find_market_cycles(sp500_df)
bear_bottoms = bear_df["Lowest Date"]
bear_bottoms = [pd.to_datetime(date) for date in bear_bottoms]

# Calculate the 252-day rolling z-score for MMFI
mmfi_data = calculate_zscore(mmfi_data, "Close", 252)

# Filter S&P 500 data to match MMFI data timeframe
sp500_df_filtered = sp500_df[sp500_df.index >= mmfi_data.index[0]]

# Create a figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

# Plot S&P 500 and MMFI on the top subplot
ax1.plot(sp500_df_filtered.index, sp500_df_filtered["Close"], label="S&P 500")
ax1.set_title("S&P 500 and MMFI")
ax1.set_ylabel("Price")

# Set the x limit
ax1.set_xlim([mmfi_data.index[0], mmfi_data.index[-1]])

# Plot the MMFI on the bottom subplot
ax2.plot(mmfi_data.index, mmfi_data["Close"], color="purple", label="MMFI")
ax2.set_xlabel("Date")
ax2.set_ylabel("Value")

# Filter bear_bottoms to match MMFI data timeframe
relevant_bear_bottoms = [bottom for bottom in bear_bottoms if bottom >= mmfi_data.index[0] and bottom <= mmfi_data.index[-1]]

# Add vertical lines for each bear market bottom to the S&P 500 plot
bear_bottom_zscores = []
for date in relevant_bear_bottoms:
    ax1.axvline(x=date, color="r", linestyle=":", alpha=0.7)
    
    # Find the closest date in mmfi_data to the bear end date
    closest_date = mmfi_data.index[mmfi_data.index.get_indexer([date], method="nearest")[0]]

    # Get the MMFI at that date
    mmfi = mmfi_data.loc[closest_date, "Close"]
    
    # Get the z-score at that date
    z_score = mmfi_data.loc[closest_date, "Close Z-Score"]
    bear_bottom_zscores.append((date, closest_date, mmfi, z_score))

# Add vertical lines when MMFI is below 10
for date in mmfi_data.index[mmfi_data["Close"] <= 10]:
    ax2.axvline(x=date, color="green", linestyle="--", alpha=0.5)

# Manually add a line for the legend
custom_lines = [Line2D([0], [0], color="r", linestyle=":", alpha=0.7), 
                 Line2D([0], [0], color="green", linestyle="--", alpha=0.5)]
# Custom labels for the legend
custom_labels = ["Bear Market Bottom", "MMFI < 10"]

# Get existing handles and labels
handles, labels = ax1.get_legend_handles_labels()
handles.extend(custom_lines)
labels.extend(custom_labels)

ax1.legend(handles, labels)
ax2.legend()

plt.tight_layout()
plt.show()

# Print the bear end dates and their corresponding z-scores
print("Bear Market Bottoms and MMFI Data:")
for date, closest_date, mmfi, z_score in bear_bottom_zscores:
    print(f"Bear Market End Date: {date}, Closest MMFI Date: {closest_date}, MMFI: {mmfi}, Z-Score: {z_score}")

In [ ]:
# Create the trading strategy based on MMTH and MMFI signals
def create_trading_signals(mmth_data, mmfi_data):
    signals = pd.DataFrame(index=mmth_data.index)
    signals['MMTH'] = mmth_data['Close']
    
    # Align MMFI data with MMTH data
    signals['MMFI'] = mmfi_data['Close'].reindex(mmth_data.index, method='ffill')
    
    # Create lagged values to detect changes
    signals['MMTH_prev'] = signals['MMTH'].shift(1)
    signals['MMFI_prev'] = signals['MMFI'].shift(1)
    
    # Define trading conditions using vectorized operations
    buy_conditions = (
        ((signals['MMTH_prev'] < 50) & (signals['MMTH'] > 50)) |  # MMTH crosses above 50
        ((signals['MMTH_prev'] > 10) & (signals['MMTH'] < 10)) |  # MMTH crosses below 10
        ((signals['MMFI_prev'] > 10) & (signals['MMFI'] < 10))    # MMFI crosses below 10
    )

    sell_conditions = (signals['MMTH_prev'] > 50) & (signals['MMTH'] < 50)  # MMTH crosses below 50

    # Initialize signal column
    signals['Signal'] = None
    signals.loc[buy_conditions, 'Signal'] = 'BUY'
    signals.loc[sell_conditions, 'Signal'] = 'SELL'
    
    # Vectorized position calculation
    signal_values = signals['Signal'].fillna('HOLD')
    positions = []
    position = 0
    trade_log = []
    
    for date, row in signals.iterrows():
        if pd.isna(row['Signal']):
            positions.append(position)
            continue
            
        if row['Signal'] == 'BUY' and position == 0:
            position = 1
            trade_log.append({
                'Date': date, 
                'Action': 'BUY', 
                'MMTH': row['MMTH'], 
                'MMFI': row['MMFI']
            })
        elif row['Signal'] == 'SELL' and position == 1:
            position = 0
            trade_log.append({
                'Date': date, 
                'Action': 'SELL', 
                'MMTH': row['MMTH'], 
                'MMFI': row['MMFI']
            })
        
        positions.append(position)
    
    signals['Position'] = positions
    return signals, trade_log

# Generate trading signals
trading_signals, trade_log = create_trading_signals(mmth_data, mmfi_data)

# Vectorized performance calculation
sp500_strategy = sp500_df.reindex(trading_signals.index, method='ffill')

# Calculate returns using vectorized operations
sp500_returns = sp500_strategy['Close'].pct_change().fillna(0)
positions_shifted = trading_signals['Position'].shift(1).fillna(0)

# Strategy returns based on position
strategy_returns = sp500_returns * positions_shifted
strategy_returns = strategy_returns.iloc[1:]  # Remove first NaN
sp500_returns_strategy = sp500_returns.iloc[1:]

# Create performance DataFrame
performance_df = pd.DataFrame({
    'Strategy_Return': strategy_returns,
    'SP500_Return': sp500_returns_strategy
}, index=strategy_returns.index)

# Calculate cumulative returns
performance_df['Strategy_Cumulative'] = (1 + performance_df['Strategy_Return']).cumprod()
performance_df['SP500_Cumulative'] = (1 + performance_df['SP500_Return']).cumprod()

# Optimized plotting with better performance
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Main plot - Equity curves
axes[0].plot(performance_df.index, performance_df['Strategy_Cumulative'], 
             label='MMTH/MMFI Strategy', linewidth=2, color='blue')
axes[0].plot(performance_df.index, performance_df['SP500_Cumulative'], 
             label='S&P 500 Buy & Hold', linewidth=2, color='red')
axes[0].set_title('MMTH/MMFI Strategy vs S&P 500 Buy & Hold', fontsize=14)
axes[0].set_ylabel('Cumulative Return')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot MMTH with signals
axes[1].plot(trading_signals.index, trading_signals['MMTH'], label='MMTH', color='purple')
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.7, label='MMTH = 50')
axes[1].axhline(y=10, color='green', linestyle='--', alpha=0.7, label='MMTH = 10')

# Efficient signal plotting
buy_mask = trading_signals['Signal'] == 'BUY'
sell_mask = trading_signals['Signal'] == 'SELL'

if buy_mask.any():
    buy_data = trading_signals[buy_mask]
    axes[1].scatter(buy_data.index, buy_data['MMTH'], 
                   color='green', marker='^', s=50, label='Buy Signal')

if sell_mask.any():
    sell_data = trading_signals[sell_mask]
    axes[1].scatter(sell_data.index, sell_data['MMTH'], 
                   color='red', marker='v', s=50, label='Sell Signal')

axes[1].set_title('MMTH with Trading Signals')
axes[1].set_ylabel('MMTH Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot MMFI
axes[2].plot(trading_signals.index, trading_signals['MMFI'], label='MMFI', color='orange')
axes[2].axhline(y=10, color='blue', linestyle='--', alpha=0.7, label='MMFI = 10')

# Mark MMFI-specific buy signals
mmfi_buy_mask = buy_mask & (trading_signals['MMFI'] <= 10)
if mmfi_buy_mask.any():
    mmfi_buy_data = trading_signals[mmfi_buy_mask]
    axes[2].scatter(mmfi_buy_data.index, mmfi_buy_data['MMFI'], 
                   color='green', marker='^', s=50, label='MMFI Buy Signal')

axes[2].set_title('MMFI with Trading Signals')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('MMFI Value')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Optimized performance metrics calculation
def calculate_performance_metrics(returns, cumulative_returns):
    total_return = cumulative_returns.iloc[-1] - 1
    annual_return = cumulative_returns.iloc[-1] ** (252 / len(returns)) - 1
    volatility = returns.std() * np.sqrt(252)
    sharpe = annual_return / volatility if volatility > 0 else 0
    
    # Maximum drawdown calculation
    peak = cumulative_returns.expanding().max()
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = drawdown.min()
    
    return {
        'total_return': total_return,
        'annual_return': annual_return,
        'volatility': volatility,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown
    }

# Calculate metrics
strategy_metrics = calculate_performance_metrics(
    performance_df['Strategy_Return'], 
    performance_df['Strategy_Cumulative']
)
sp500_metrics = calculate_performance_metrics(
    performance_df['SP500_Return'], 
    performance_df['SP500_Cumulative']
)

# Display results
print("Performance Summary:")
print(f"Strategy Total Return: {strategy_metrics['total_return']:.2%}")
print(f"S&P 500 Total Return: {sp500_metrics['total_return']:.2%}")
print(f"Strategy Annual Return: {strategy_metrics['annual_return']:.2%}")
print(f"S&P 500 Annual Return: {sp500_metrics['annual_return']:.2%}")
print(f"Strategy Volatility: {strategy_metrics['volatility']:.2%}")
print(f"S&P 500 Volatility: {sp500_metrics['volatility']:.2%}")
print(f"Strategy Sharpe Ratio: {strategy_metrics['sharpe']:.4f}")
print(f"S&P 500 Sharpe Ratio: {sp500_metrics['sharpe']:.4f}")
print(f"Strategy Max Drawdown: {strategy_metrics['max_drawdown']:.2%}")
print(f"S&P 500 Max Drawdown: {sp500_metrics['max_drawdown']:.2%}")

print(f"\nTrade Log ({len(trade_log)} trades):")
for trade in trade_log[-10:]:  # Show last 10 trades
    print(f"{trade['Date'].strftime('%Y-%m-%d')}: {trade['Action']} - "
          f"MMTH: {trade['MMTH']:.2f}, MMFI: {trade['MMFI']:.2f}")

In [ ]:
# Calculate daily returns for S&P 500
sp500_df = get_df("^GSPC", current_date, adj=True, max_period=True)
sp500_df["Percent Change"] = sp500_df["Close"].pct_change()

# Extract day of week and month from index
sp500_df["Day of Week"] = sp500_df.index.dayofweek
sp500_df["Month"] = sp500_df.index.month

# Calculate monthly returns
monthly_returns = sp500_df["Close"].resample("ME").last().pct_change().dropna()
monthly_returns_with_month = pd.DataFrame({
    "Return": monthly_returns,
    "Month": monthly_returns.index.month
})

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Daily returns by day of week (Monday to Friday)
weekday_labels = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

# Vectorized calculation for weekday means
weekday_data = sp500_df[sp500_df["Day of Week"] < 5].groupby("Day of Week")["Percent Change"].mean()
weekday_means = [weekday_data.get(day, 0) for day in range(5)]

ax1.bar(weekday_labels, weekday_means, alpha=0.7)
ax1.set_title("Average Daily Returns by Day of Week")
ax1.set_xlabel("Day of Week")
ax1.set_ylabel("Average Daily Return")

# Plot 2: Monthly returns by month (Jan to Dec)
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", 
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Vectorized calculation for monthly means
monthly_data = monthly_returns_with_month.groupby("Month")["Return"].mean()
monthly_means = [monthly_data.get(month, 0) for month in range(1, 13)]

ax2.bar(month_labels, monthly_means, alpha=0.7)
ax2.set_title("Average Monthly Returns by Month")
ax2.set_xlabel("Month")
ax2.set_ylabel("Average Monthly Return")

plt.tight_layout()
plt.show()

# Print summary statistics using vectorized operations
print("Daily Returns by Day of Week:")
weekday_stats = sp500_df[sp500_df["Day of Week"] < 5].groupby("Day of Week")["Percent Change"].agg(["mean", "std", "count"])
for i, day in enumerate(weekday_labels):
    if i in weekday_stats.index:
        stats = weekday_stats.loc[i]
        print(f"{day}: Mean={stats['mean']:.4f}, Std={stats['std']:.4f}, Count={stats['count']}")

print("\nMonthly Returns by Month:")
monthly_stats = monthly_returns_with_month.groupby("Month")["Return"].agg(["mean", "std", "count"])
for i, month in enumerate(month_labels):
    month_num = i + 1
    if month_num in monthly_stats.index:
        stats = monthly_stats.loc[month_num]
        print(f"{month}: Mean={stats['mean']:.4f}, Std={stats['std']:.4f}, Count={stats['count']}")

In [ ]:
# Define return periods in trading days
return_periods = {
    "1W": pd.Timedelta(days=5),
    "1M": pd.Timedelta(days=21),
    "3M": pd.Timedelta(days=63),
    "6M": pd.Timedelta(days=126),
    "1Y": pd.Timedelta(days=252)
}

# Find all MMFI signals <= 10 at least one week apart
signals = mmfi_data.index[mmfi_data["Close"] <= 10]
filtered_signals = signals[(signals.to_series().diff().dt.days.isna()) | (signals.to_series().diff().dt.days > 7)]

# Compute SP500 returns after each signal
def get_returns(dt):
    row = {"Date": dt.date()}
    if dt in sp500_df.index:
        start = sp500_df.at[dt, "Close"]
        for label, delta in return_periods.items():
            future_date = dt + delta
            # Find the closest trading day on or after the target date
            future_idx = sp500_df.index.searchsorted(future_date)
            if future_idx < len(sp500_df):
                end = sp500_df.iloc[future_idx]["Close"]
                row[label] = end / start - 1
            else:
                row[label] = None
    else:
        for label in return_periods:
            row[label] = None
    return row

mmfi_sp500_returns_df = pd.DataFrame([get_returns(dt) for dt in filtered_signals]).set_index("Date")

# Calculate the average returns for all signals
avg_row = mmfi_sp500_returns_df.mean().to_frame().T
avg_row.index = ["Average"]

# Append the average row to the DataFrame
mmfi_sp500_returns_df = pd.concat([mmfi_sp500_returns_df, avg_row])
display(mmfi_sp500_returns_df)

In [ ]:
# Get the price data of QQQ, TQQQ, and VIX
qqq_df = get_df("^IXIC", current_date)
tqqq_df = get_df("TQQQ", current_date)
vix_df = get_df("^VIX", current_date)

# Filter qqq_df to match the starting date of TQQQ
qqq_df_tqqq = qqq_df[qqq_df.index >= tqqq_df.index[0]]

# Calculate the CAGR of TQQQ
CAGR_tqqq = (tqqq_df["Close"].iloc[-1] / tqqq_df["Close"].iloc[0])**(1 / (len(tqqq_df) / 252)) - 1

# Calculate the CAGR of QQQ
CAGR_qqq = (qqq_df_tqqq["Close"].iloc[-1] / qqq_df_tqqq["Close"].iloc[0])**(1 / (len(qqq_df_tqqq) / 252)) - 1

# Calculate the CAGR ratio of TQQQ to QQQ
tqqq_factor = CAGR_tqqq / CAGR_qqq

# Calculate the percentage change of QQQ
qqq_df["Percent Change"] = qqq_df["Close"].pct_change()

# Calculate the closing prices of TQQQ
qqq_df["TQQQ Percent Change"] = qqq_df["Percent Change"] * tqqq_factor
qqq_df["TQQQ Close"] = (1 + qqq_df["TQQQ Percent Change"]).cumprod()

# Create the mock TQQQ dataframe
tqqq_mock_df = qqq_df[["TQQQ Close", "TQQQ Percent Change"]].copy()

# Copy the index from qqq_df
tqqq_mock_df.index = qqq_df.index

# Scale the mock TQQQ closing prices to match the most recent TQQQ close
tqqq_mock_df["TQQQ Close"] = tqqq_mock_df["TQQQ Close"] * (tqqq_df["Close"].iloc[-1] / tqqq_mock_df["TQQQ Close"].iloc[-1])
tqqq_mock_df.rename(columns={"TQQQ Close": "Close", "TQQQ Percent Change": "Percent Change"}, inplace=True)

In [ ]:
show = 252 * 3
stocks = ["GC=F", "SI=F", "HG=F"]
dfs = [get_df(stock, current_date, redownload=True) for stock in stocks]
metal_df = merge_stocks(stocks, dfs)
metal_df["Gold/Silver Ratio"] = metal_df["Close (GC=F)"] / metal_df["Close (SI=F)"]
metal_df["Gold/Copper Ratio"] = metal_df["Close (GC=F)"] / metal_df["Close (HG=F)"]

# Restrict the dataframe
metal_df = metal_df[-show:]

# Create a figure with two subplots: one for the metal prices, one for the ratios
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

# Plot the metal prices on the first subplot
close_goldfirst = metal_df["Close (GC=F)"].iloc[0]
close_silverfirst = metal_df["Close (SI=F)"].iloc[0]
close_copperfirst = metal_df["Close (HG=F)"].iloc[0]
ax1.plot(100 / close_goldfirst * metal_df["Close (GC=F)"], label="Gold (scaled)", color="gold")
ax1.plot(100 / close_silverfirst * metal_df["Close (SI=F)"], label="Silver (scaled)", color="silver")
ax1.plot(100 / close_copperfirst * metal_df["Close (HG=F)"], label="Copper (scaled)", color="peru")

# Set the label of the first subplot
ax1.set_ylabel("Price")

# Set the x limit of the first subplot
ax1.set_xlim(metal_df.index[0], metal_df.index[-1])

# Plot the ratios on the second subplot
goldsilver_ratio_first = metal_df["Gold/Silver Ratio"].iloc[0]
goldcopper_ratio_first = metal_df["Gold/Copper Ratio"].iloc[0]
ax2.plot(100 / goldsilver_ratio_first * metal_df["Gold/Silver Ratio"], color="silver", label="Gold/Silver Ratio")
ax2.plot(100 / goldcopper_ratio_first * metal_df["Gold/Copper Ratio"], color="peru", label="Gold/Copper Ratio")

# Set the y label of the second subplot
ax2.set_ylabel("Ratio wrt Gold")

# Set the x label
plt.xlabel("Date")

# Set the title
plt.suptitle(f"Metal prices comparison")

# Combine the legends and place them at the top subplot
handles, labels = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
handles += handles2
labels += labels2
ax1.legend(handles, labels)

# Adjust the spacing between subplots
plt.tight_layout()

# Save the plot
plt.savefig("Result/Figure/metalcf.png", dpi=300)    

# Show the plot
plt.show()